# Project SD-08 — Invoice RAG

> Goal: Turn a PDF invoice into a searchable RAG corpus without losing its
> structure — extract text with pdfplumber, recognize the header block, line
> items, and totals as typed units, and let retrieval answer questions about
> amounts, dates, and customers.

```
Loader      : pdfplumber (structure-aware: header → header, line items → line_item, totals → totals)
Splitter    : RecursiveCharacterTextSplitter (safety net — units stay atomic)
Embedding   : fastembed / BAAI/bge-base-en-v1.5 (local, 768-dim)
Vector DB   : Chroma
Retriever   : Similarity Search (Top-K) + metadata filter
Prompt      : Basic Context + Question
LLM         : Ollama qwen2.5-coder:7b (local)
```

Learn:

* Why a PDF is *not* a text file — pages are drawings, text is optional
* Why scanned/image-only PDFs yield zero text and need OCR
* How a structure-aware pass turns an invoice into header / line_item / totals units
* How `invoice_id` metadata enables filtering to one invoice


## The naive way (what breaks)

A PDF is a page-layout format: it draws glyphs at coordinates, it does not
store paragraphs. Text extraction tools like `pdfplumber` and `PyPDFLoader`
work by reading the text operators in the content stream — which only exist if
the PDF was *generated* by a program. A scanned invoice is just a photo glued
onto a page: there is no text layer, so extraction returns nothing.

The tempting shortcut is to flatten every page like a text file:

```python
text = "\n".join(page.extract_text() or "" for page in pdf.pages)   # naive flatten
```

Two things break.

**1. Scanned/image-only PDFs yield zero text.** `watson-hall-1898.pdf`,
`macy-receipt.pdf`, and `szamla-minta.jpg` are images, not text. `extract_text()`
returns `None` on every page — a RAG pipeline built on them answers nothing
without OCR (Tesseract, Google Vision, …).

**2. Even a text PDF is a flat string.** `sample-invoice.pdf` *does* have a text
layer, but flattening it destroys the invoice's structure: the header block
(invoice number, customer, dates), the line-item rows, and the totals block all
become one undifferentiated blob. A character-count splitter can then separate
"TOTAL DUE" from "$610.00", or a line item from its price.

Run the cell below to watch both failures happen.


In [1]:
import os
import pdfplumber

scanned = [
    "../../../Data/SD-08-invoices/watson-hall-1898.pdf",
    "../../../Data/SD-08-invoices/macy-receipt.pdf",
]
for path in scanned:
    with pdfplumber.open(path) as pdf:
        text = "\n".join(page.extract_text() or "" for page in pdf.pages)
    print(f"{os.path.basename(path):24} → {len(text.strip())} chars of text")

with pdfplumber.open("../../../Data/SD-08-invoices/sample-invoice.pdf") as pdf:
    flat = "\n".join(page.extract_text() or "" for page in pdf.pages)
print(f"{'sample-invoice.pdf':24} → {len(flat.strip())} chars of text")


watson-hall-1898.pdf     → 0 chars of text
macy-receipt.pdf         → 0 chars of text
sample-invoice.pdf       → 748 chars of text


## 0 · Setup — a fully local stack

**WHAT:** Imports the whole RAG stack plus `pdfplumber`. Everything runs on
this machine: `fastembed` (`BAAI/bge-base-en-v1.5`) produces the embeddings
and Ollama (`qwen2.5-coder:7b`) generates the answers. No API key, no rate
limits.

**WHY:** `pdfplumber` is already in `requirements.txt` (Special Documents
series), so there is no install cell. Embeddings and generation are local for
the same reason the SD-01 strategy-comparison notebook moved off Gemini: the
free tier caps embedding at roughly 100 requests/minute, and retrying a 429
burns more of the same exhausted quota — a retry storm that never recovers.
Local ONNX embedding is ~0.3s per 100 chunks on CPU.

**WHAT TO EXPECT:** All imports succeed, and a local stack check confirms
Ollama is running with `qwen2.5-coder:7b` available.


In [2]:
import requests

try:
    tags = requests.get("http://localhost:11434/api/tags", timeout=5).json()
    models = [m["name"] for m in tags.get("models", [])]
    print("Ollama running, models:", ", ".join(models) or "(none pulled)")
    assert "qwen2.5-coder:7b" in models, "pull qwen2.5-coder:7b first"
except Exception as e:
    print(f"Ollama not reachable at localhost:11434 — {e}")
    print("Install Ollama (ollama.com) and run: ollama pull qwen2.5-coder:7b")


Ollama running, models: qwen2.5-coder:7b


In [3]:
import pdfplumber
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_ollama import ChatOllama
from fastembed import TextEmbedding


/home/magus/.pyenv/versions/magus/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1 · Load — pdfplumber, structure-aware

**WHAT:** `invoice_to_docs()` does two passes over the PDF. First it extracts
the text of every page with `pdfplumber` (page-by-page, so page boundaries are
preserved). Then a structure-aware pass reads the raw lines and classifies them
into units:

* the **header block** — invoice number, customer, dates, customer ID
* the **line-item rows** — each `QUANTITY / DESCRIPTION / UNIT PRICE / TOTAL` row
* the **totals block** — SUBTOTAL, SALES TAX, TOTAL, PREVIOUS BALANCE, TOTAL DUE

Every unit becomes one langchain `Document` carrying `source`, `type`
(`header` / `line_item` / `totals`), and `invoice_id` metadata.

**WHY:** This is the block SD-08 changes. An invoice is a *form*, not prose —
an amount only means something next to its label. Keeping the header, line
items, and totals as separate typed units means retrieval can return "TOTAL DUE
$610.00" as a complete fact, and the `invoice_id` metadata enables filtering to
one invoice later.

**WHAT TO EXPECT:** The fixture — `sample-invoice.pdf` (INV-100, MICROSOFT
CORPORATION) — yields a handful of units: one header block, one line item
("Test for 23 fields", qty 1, unit price 1), and one totals block (SUBTOTAL
$100.00, SALES TAX $10.00, TOTAL $110.00, PREVIOUS BALANCE $500.00, TOTAL DUE
$610.00). Metadata never contains `None` (Chroma rejects it).


In [4]:
import os

PDF_PATH = "../../../Data/SD-08-invoices/sample-invoice.pdf"

if not os.path.exists(PDF_PATH):
    print(f"Sample file not found: {PDF_PATH}")
    print("Look for it under Data/SD-08-invoices/.")
else:
    print(f"Found sample: {PDF_PATH}")


Found sample: ../../Data/SD-08-invoices/sample-invoice.pdf


In [5]:
def extract_pdf_text(path):
    """Return one string per page, extracted with pdfplumber."""
    pages = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            pages.append(page.extract_text() or "")
    return pages


In [6]:
def parse_header_lines(lines):
    """Header block: invoice number, customer, dates, customer ID."""
    keys = ("INVOICE:", "DATE:", "DUE DATE:", "CUSTOMER NAME:", "CUSTOMER ID:")
    return [ln for ln in lines if any(k in ln for k in keys)]


In [7]:
def parse_line_items(lines):
    """Rows between the QUANTITY column header and the SUBTOTAL line."""
    items = []
    for i, ln in enumerate(lines):
        if ln.startswith("QUANTITY"):
            for row in lines[i + 1:]:
                if row.startswith("SUBTOTAL"):
                    break
                items.append(row)
            break
    return items


In [8]:
def parse_totals(lines):
    """Totals block: SUBTOTAL, SALES TAX, TOTAL, PREVIOUS BALANCE, TOTAL DUE."""
    keys = ("SUBTOTAL", "SALES TAX", "TOTAL", "PREVIOUS BALANCE", "TOTAL DUE")
    return [ln for ln in lines if ln.startswith(keys)]


In [9]:
def invoice_to_docs(path, invoice_id):
    """Structure-aware pass: header / line_item / totals → Documents."""
    lines = [ln.strip() for page in extract_pdf_text(path) for ln in page.splitlines()]
    meta = {"source": path, "invoice_id": invoice_id}

    header = parse_header_lines(lines)
    items = parse_line_items(lines)
    totals = parse_totals(lines)

    docs = [Document(page_content="\n".join(header), metadata={**meta, "type": "header"})]
    docs += [Document(page_content=row, metadata={**meta, "type": "line_item"}) for row in items]
    docs += [Document(page_content="\n".join(totals), metadata={**meta, "type": "totals"})]
    return docs


In [10]:
docs = invoice_to_docs(PDF_PATH, invoice_id="INV-100")

print(f"parsed {len(docs)} structural units")
for d in docs:
    print(f"  [{d.metadata.get('type'):9}] {d.page_content[:50]!r}")


parsed 3 structural units
  [header   ] 'Contoso Headquarters INVOICE: INV-100\n123 456th St'
  [line_item] '1 Test for 23 fields 1 $100.00'
  [totals   ] 'SUBTOTAL $100.00\nSALES TAX $10.00\nTOTAL $110.00\nPR'


## 2 · Split — units are already atomic

**WHAT:** Each unit from the load pass is already a small, complete fact: the
header block, one line item, or the totals block. `RecursiveCharacterTextSplitter`
runs as a *safety net* with a large `chunk_size` (1000) and zero overlap — large
enough that no unit is ever cut, so the pipeline still has a splitter step but
it never destroys structure.

**WHY:** The naive splitter slices by character count and can separate "TOTAL
DUE" from "$610.00". Here structure was decided at load time, so the splitter
must be a no-op, not a second opinion. This is the same lesson as SD-01 applied
to invoices: the parser decides the chunk boundaries, not character counts.

**WHAT TO EXPECT:** One chunk per unit — the chunk count equals the unit count.


In [11]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
chunks = splitter.split_documents(docs)

print(f"{len(docs)} units → {len(chunks)} chunks (splitter is a no-op)")
for c in chunks:
    print(f"  [{c.metadata.get('type'):9}] {c.page_content[:45]!r}")


3 units → 3 chunks (splitter is a no-op)
  [header   ] 'Contoso Headquarters INVOICE: INV-100\n123 456'
  [line_item] '1 Test for 23 fields 1 $100.00'
  [totals   ] 'SUBTOTAL $100.00\nSALES TAX $10.00\nTOTAL $110.'


## 3 · Embed — text → vectors (local)

**WHAT:** A small `LocalEmbeddings` adapter wraps `fastembed` with the
`BAAI/bge-base-en-v1.5` model (768 dimensions, runs on CPU). Every chunk
becomes a numeric vector; similar content lands close together in vector
space.

**WHY:** Embeddings make retrieval semantic: "what do we owe" can match the
"TOTAL DUE $610.00" chunk even when the wording differs. Local ONNX embedding
needs no API key and has no rate limits — 100 chunks embed in ~0.3s on CPU —
unlike Gemini's free tier, which caps at ~100 embed requests/minute.

**WHAT TO EXPECT:** An embeddings object, then a 768-dimensional vector from
a sample query.


In [12]:
class LocalEmbeddings:
    """LangChain-compatible local embedder (fastembed / BAAI/bge-base-en-v1.5, 768-dim).

    No API key, no rate limits: 100 chunks embed in ~0.3s on CPU.
    Implements the langchain Embeddings interface (embed_documents +
    embed_query) so Chroma.from_documents accepts it directly.
    """
    MODEL = "BAAI/bge-base-en-v1.5"

    def __init__(self):
        self._emb = TextEmbedding(model_name=self.MODEL)

    def embed_documents(self, texts):
        return [v.tolist() for v in self._emb.embed(texts, batch_size=64)]

    def embed_query(self, text):
        return next(self._emb.query_embed(text)).tolist()


embeddings = LocalEmbeddings()

sample_vec = embeddings.embed_query("What's the total?")
print(len(sample_vec), "dimensions per chunk")


768 dimensions per chunk


## 4 · Store — index the vectors in Chroma

**WHAT:** `Chroma.from_documents(documents=chunks, embedding=embeddings)`
embeds all chunks and writes them into a Chroma collection. A
`chroma_langchain_db/` folder appears next to the notebook.

**WHY:** The vector store is the pipeline's memory: retrieval searches this
index in milliseconds instead of re-reading the PDF.

**WHAT TO EXPECT:** A `Chroma` object (and the folder on disk). Because we
dropped `None` from metadata, every stored row is Chroma-safe.


In [13]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
)


## 5 · Retrieve — top-k similarity search

**WHAT:** `vector_store.similarity_search(query, k=3)` embeds the question and
returns the 3 closest chunks. We join them into a `context` string for the
prompt.

**WHY:** This is the "R" in RAG. Because our chunks are whole header / line /
totals units, a retrieved hit is a complete fact — the model sees "TOTAL DUE
$610.00" as a unit, not a fragment.

**WHAT TO EXPECT:** 3 chunks whose `type` metadata tells you what kind of
evidence was found (header vs line_item vs totals).


In [14]:
query = "What is the total due on this invoice?"
retrieved = vector_store.similarity_search(query, k=3)

print(f"retrieved {len(retrieved)} chunks for: {query!r}")


retrieved 3 chunks for: 'What is the total due on this invoice?'


In [15]:
context = "\n\n".join(d.page_content for d in retrieved)

for d in retrieved:
    print(f"[{d.metadata.get('type')}] invoice={d.metadata.get('invoice_id')} → {d.page_content[:60]!r}")


[totals] invoice=INV-100 → 'SUBTOTAL $100.00\nSALES TAX $10.00\nTOTAL $110.00\nPREVIOUS BAL'
[header] invoice=INV-100 → 'Contoso Headquarters INVOICE: INV-100\n123 456th St DATE: 11/'
[line_item] invoice=INV-100 → '1 Test for 23 fields 1 $100.00'


In [16]:
filtered = vector_store.similarity_search(
    "What is the total?",
    k=3,
    filter={"invoice_id": "INV-100"},
)

print(f"filtered to INV-100 → {len(filtered)} chunks")
for d in filtered:
    print(f"  [{d.metadata.get('type'):9}] {d.page_content[:50]!r}")


filtered to INV-100 → 3 chunks
  [totals   ] 'SUBTOTAL $100.00\nSALES TAX $10.00\nTOTAL $110.00\nPR'
  [line_item] '1 Test for 23 fields 1 $100.00'
  [header   ] 'Contoso Headquarters INVOICE: INV-100\n123 456th St'


## 6 · Prompt — package context + question

**WHAT:** A `ChatPromptTemplate` wraps the instruction "answer using ONLY the
provided context" (with an explicit "I don't know" fallback) around the joined
`context` and the `question`.

**WHY:** The prompt is the contract that stops hallucination — the model may
only answer from the retrieved chunks. Reading the rendered `messages` shows
exactly what the model sees.

**WHAT TO EXPECT:** A `ChatPromptTemplate`, then the rendered `messages`.


In [17]:
template = """You are a helpful assistant.

Answer the question using ONLY the provided context.
If the answer is not contained in the context, say:
"I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
"""
prompt = ChatPromptTemplate.from_template(template)


In [18]:
messages = prompt.invoke({"context": context, "question": query})
print(messages)


messages=[HumanMessage(content='You are a helpful assistant.\n\nAnswer the question using ONLY the provided context.\nIf the answer is not contained in the context, say:\n"I don\'t know based on the provided context."\n\nContext:\nSUBTOTAL $100.00\nSALES TAX $10.00\nTOTAL $110.00\nPREVIOUS BALANCE $500.00\nTOTAL DUE $610.00\n\nContoso Headquarters INVOICE: INV-100\n123 456th St DATE: 11/15/2019\nNew York, NY, 10001 DUE DATE: 12/15/2019\nCUSTOMER NAME: MICROSOFT CORPORATION\nCUSTOMER ID: CID-12345\n\n1 Test for 23 fields 1 $100.00\n\nQuestion:\nWhat is the total due on this invoice?\n\nAnswer:\n', additional_kwargs={}, response_metadata={})]


## 7 · Answer — a local LLM reads the prompt

**WHAT:** `ChatOllama(model="qwen2.5-coder:7b")` invokes the filled
`messages` against a model running on your machine; the answer is printed.

**WHY:** The final block. The model reads the retrieved evidence plus the
question and produces a grounded answer — grounded in the invoice, not in its
own memory. Ollama keeps the whole pipeline offline; the first call loads the
model into RAM, so expect a few seconds of latency.

**WHAT TO EXPECT:** A natural-language answer that reflects the parsed
header/amount/totals content.


In [19]:
llm = ChatOllama(model="qwen2.5-coder:7b", temperature=0)
response = llm.invoke(messages)
print(response.content)


The total due on this invoice is $610.00.


## 8 · Try it yourself — your sandbox

Change the question, change `k`, or point the notebook at your own invoice PDF.
Two question shapes worth testing, both answered from retrieval:

* `"What is the total due on this invoice?"` — totals query
* `"Who is the customer and what is their ID?"` — header query


In [20]:
query = "What is the total due on this invoice?"
hits = vector_store.similarity_search(query, k=3)
ctx = "\n\n".join(d.page_content for d in hits)
print(llm.invoke(prompt.invoke({"context": ctx, "question": query})).content)


The total due on this invoice is $610.00.


In [21]:
query = "Who is the customer on this invoice and what is their ID?"
hits = vector_store.similarity_search(query, k=3)
ctx = "\n\n".join(d.page_content for d in hits)
print(llm.invoke(prompt.invoke({"context": ctx, "question": query})).content)


The customer on this invoice is MICROSOFT CORPORATION, and their ID is CID-12345.


## What you should notice

* **The changed block is the parser, not the pipeline.** Load/split/embed/store/
  retrieve/prompt/answer keep the same shape as the earlier projects — only how
  the PDF becomes text is new, and embedding + generation now run locally
  (fastembed + Ollama) with no API key.
* **A PDF is a page layout, not a text file.** Text extraction only works when a
  text layer exists; scanned/image-only files (`watson-hall-1898.pdf`,
  `macy-receipt.pdf`, `szamla-minta.jpg`) return zero text and need OCR.
* **An invoice is a form, not prose.** Flattening it to one string destroys the
  meaning of every amount — a total only means something next to its label.
* **Structure-aware parsing beats character splitting.** Classifying header /
  line_item / totals units at load time means retrieval returns complete facts,
  and the splitter becomes a no-op safety net.
* **Metadata enables filtering.** `invoice_id` on every chunk lets you query one
  invoice with `filter={"invoice_id": "INV-100"}` — the same idea scales to a
  whole folder of invoices.
* **Metadata must be Chroma-safe.** `None` values are rejected, so metadata is
  built without empty keys.


## Exercises

1. **Point the parser at another invoice.** Drop one of the other PDFs in
   `Data/SD-08-invoices/` (e.g. `Invoice_1.pdf`) into `PDF_PATH`, re-run the
   pipeline, and watch the header / line_item / totals units change.
2. **Add OCR for scanned invoices.** Install `pytesseract`, render
   `watson-hall-1898.pdf` pages to images, and run Tesseract on them — then feed
   the OCR text through the same structure-aware pass and compare retrieval.
3. **Scale to a folder of invoices.** Loop over every PDF in
   `Data/SD-08-invoices/`, tag each chunk with its own `invoice_id`, and ask a
   question that needs `filter={"invoice_id": …}` to pick the right invoice.
